<a href="https://colab.research.google.com/github/oninonins/Machine-learning-/blob/main/JS06%20-%20Regresi/praktikum%202.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Praktikum 2 — Support Vector Regression (SVR)

Berbeda dengan Regresi Linier yang hanya memodelkan hubungan lurus, **SVR** dengan kernel RBF (Radial Basis Function) bisa mengamodelkan hubungan **non-linear** antara tingkat posisi dan gaji.

Dataset `Posisi_gaji.csv` berisi 10 posisi dengan `Level` (tingkat posisi) dan `Gaji`.

Tahapan sesuai modul:
1. Import library · 2. Import dataset · 3. Feature scaling · 4. Fitting SVR kernel RBF · 5. Visualisasi hasil · 6. Prediksi · 7. Menampilkan hasil · 8. Validasi · 9. Evaluasi model (MAE, MSE, RMSE, R²).

## Langkah 1-2 — Import library dan dataset

In [ ]:
# Mengimpor library
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# File CSV tersedia di folder repo "JS06 - Regresi".
# Di Colab: upload lewat panel File (ikon folder) ke working directory /content, lalu jalankan cell ini.
from pathlib import Path

if not Path('Posisi_gaji.csv').exists():
    raise FileNotFoundError(
        "Posisi_gaji.csv belum ada di working directory. "
        "Upload lewat panel File di Colab (file tersedia di folder repo 'JS06 - Regresi')."
    )
print("Posisi_gaji.csv siap:", Path('Posisi_gaji.csv').stat().st_size, "byte")

# Mengimpor dataset (pastikan file CSV 'Posisi_gaji.csv' ada di direktori yang sama)
dataset = pd.read_csv('Posisi_gaji.csv')
dataset.head()

# X = tingkat posisi (kolom Level), y = gaji (kolom Gaji, dijadikan satu kolom)
X = dataset.iloc[:, 1:2].values
y = dataset.iloc[:, 2].values

print("Jumlah data:", len(dataset), "| kolom:", list(dataset.columns))

## Langkah 3 — Feature Scaling

SVR sangat sensitif terhadap skala data: kernel RBF menghitung jarak, dan fitur dengan skala besar akan mendominasi jarak tersebut. Karena itu **X dan y keduanya** diskalakan dengan `StandardScaler`.

`StandardScaler` mengubah nilai menjadi `(nilai - rata-rata) / standar deviasi`, sehingga hasilnya berpusat di 0 dengan-spread seragam.

In [ ]:
# Feature Scaling
from sklearn.preprocessing import StandardScaler

sc_X = StandardScaler()
sc_y = StandardScaler()

X = sc_X.fit_transform(X.reshape(-1, 1))
y = sc_y.fit_transform(y.reshape(-1, 1))

print("X (scaled) 5 baris pertama:\n", X[:5])
print("y (scaled) 5 baris pertama:\n", y[:5])

## Langkah 4 — Fitting SVR dengan kernel RBF

Kernel RBF memberi each data point sebuah "lingkaran pengaruh"; titik yang jaraknya similar gets similar prediction, sehingga model bisa membentuk kurva mulus (bukan garis lurus).

In [ ]:
# Fitting SVR ke dataset
from sklearn.svm import SVR

regressor = SVR(kernel='rbf')
regressor.fit(X, y)

print("SVR kernel:", regressor.kernel, "| C:", regressor.C, "| epsilon:", regressor.epsilon)
print("Jumlah support vector:", regressor.support_vectors_.shape[0], "dari", len(dataset), "data")

## Langkah 5 — Visualisasi hasil SVR

Titik merah adalah data observasi asli (sudah diskalakan), garis biru adalah kurva prediksi SVR. `X_grid` di finely spaced 0.01 supaya kurvanya halus.

In [ ]:
# Visualisasi hasil SVR (resolusi tinggi dan kurva yang lebih halus)
X_grid = np.arange(min(X), max(X), 0.01).reshape(-1, 1)

plt.figure(figsize=(8, 5))
plt.scatter(X, y, color='red')
plt.plot(X_grid, regressor.predict(X_grid), color='blue')
plt.title('Jujur atau tidak (SVR)')
plt.xlabel('Tingkat posisi')
plt.ylabel('Gaji')
plt.grid(alpha=0.3)
plt.show()

## Langkah 6-7 — Prediksi gaji untuk tingkat posisi 6.5

Data prediksi harus **diskalakan dengan `sc_X.transform`** (bukan `fit_transform`, supaya tidak fitting ulang), lalu hasil prediksinya **dikembalikan ke skala asli** dengan `sc_y.inverse_transform`.

In [ ]:
# Prediksi hasil
# Buat array 2D yang berisi tingkat posisi yang akan diprediksi
tingkat_posisi_prediksi = np.array([[6.5]])

# Penskalaan fitur untuk data yang akan diprediksi
tingkat_posisi_prediksi = sc_X.transform(tingkat_posisi_prediksi)

# Melakukan prediksi menggunakan model SVR
gaji_prediksi = regressor.predict(tingkat_posisi_prediksi)

# Kembalikan hasil prediksi ke skala aslinya
gaji_prediksi = sc_y.inverse_transform(gaji_prediksi.reshape(-1, 1))

# Menampilkan hasil prediksi
print("Prediksi Gaji untuk Tingkat Posisi 6.5:", round(float(gaji_prediksi[0][0]), 2))

## Langkah 8-9 — Validasi dan evaluasi model

Dievaluasi dengan **MAE**, **MSE**, **RMSE**, dan **R-squared**.

Penting soal satuan: karena `y` diskalakan, error harus dibandingkan pada **skala yang sama**. Dua versi di bawah ditampilkan:
- **skala scaled** (persis seperti output modul): error dalam satuan terstandarisasi.
- **skala asli (dolar)**: prediksi dikembalikan ke satuan semula dulu, baru dibandingkan dengan data asli — inilah angka yang bisa dipakai bernalar.

R-squared bersifat invarian terhadap penskalaan linier, jadi nilainya identik di kedua versi.

In [ ]:
# Evaluasi model
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

y_pred = regressor.predict(X)

# --- (a) metrik pada skala scaled (identik dengan output modul) ---
mae = mean_absolute_error(y.ravel(), y_pred)
mse = mean_squared_error(y.ravel(), y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y.ravel(), y_pred)

print("=== Skala scaled (satuan terstandarisasi) ===")
print("MAE:", mae)
print("MSE:", mse)
print("RMSE:", rmse)
print("R-squared:", r2)

# --- (b) metrik pada skala asli (dolar) ---
y_actual_asli = sc_y.inverse_transform(y).ravel()
y_pred_asli = sc_y.inverse_transform(regressor.predict(X).reshape(-1, 1)).ravel()

mae_asli = mean_absolute_error(y_actual_asli, y_pred_asli)
mse_asli = mean_squared_error(y_actual_asli, y_pred_asli)
rmse_asli = np.sqrt(mse_asli)
r2_asli = r2_score(y_actual_asli, y_pred_asli)

print("\n=== Skala asli (dolar) ===")
print("MAE:", round(mae_asli, 2))
print("MSE:", round(mse_asli, 2))
print("RMSE:", round(rmse_asli, 2))
print("R-squared:", round(r2_asli, 6))

print("\n=== Perbandingan dengan output modul ===")
print("Modul  -> MAE 0.22299274095734414 | MSE 0.24839989293792014 | RMSE 0.4983973243687411 | R2 0.7516001070620798")
print("Hasil  -> MAE", mae, "| MSE", mse, "| RMSE", rmse, "| R2", r2)

## Analisa

1. **Kenapa X dan y sama-sama diskalakan?** Kernel RBF bekerja pada jarak. Tanpa penskalaan, `Gaji` (ribuan sampai ratusan ribu dolar) mendominasi jarak dibanding `Level` (1-10), sehingga kurva yang dibentuk model hanya mengikuti `Gaji` dan `Level` jadi tidak bermakna. Setelah diskalakan, keduanya berpusat di 0 dengan sebaran sama.
2. **Kenapa prediksi harus di-`inverse_transform`?** Model dilatih pada `y` berskala, jadi keluarannya juga berskala. Tanpa dikembalikan ke skala asli, membandingkan prediksi (berskala) dengan data asli (dolar) menghasilkan error yang tidak bermakna — R-squared bisa negatif. Kode di atas sengaja menampilkan **kedua** versi supaya perbedaannya jelas, dan R-squared-nya tetap identik karena R-squared tidak berubah oleh transformasi linier.
3. **R-squared 0.7516** berarti sekitar **75%** variasi gaji pada data ini bisa dijelaskan oleh tingkat posisi. Sisanya 25% berasal dari faktor lain (pengalaman, pendidikan, perusahaan) yang tidak ada di dataset.
4. **MAE $63.332** pada skala dolar adalah error rata-rata absolut. Bandingkan dengan rata-rata gaji $249.500, jadi rata-rata kesalahan prediksi sekitar 25% dari nilai gaji.
5. **Dataset ini hanya 10 baris**, jadi angka ini hanya untuk demonstrasi mekanisme SVR, bukan ukuran performa yang bisa digeneralisasi. Pada data real, SVR perlu `C` dan `epsilon` di-tuning (grid search) agar performanya optimal.
6. **Prediksi posisi 6.5 = $170.370** jatuh di antara Country Manager ($110.000, level 5) dan direksi berikutnya, dan masih di bawah rata-rata gaji $249.500 — wajar karena posisi 6.5 berada di bagian bawah tabel gaji.